In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

#Dictionary to save the important parameters
saved_params = {}

In [2]:
#Loading the dataset
data = pd.read_csv('/Users/tristangarcia/desktop/Network Traffic Classification/data/train.csv')
print(data.shape)
data.head()

(145854, 44)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000009,unas,-,INT,2,0,200,0,111111.1072,254,...,4,8,0,0,0,13,8,0,Analysis,1
1,0.000009,unas,-,INT,2,0,200,0,111111.1072,254,...,2,7,0,0,0,11,7,0,Analysis,1
2,0.000008,unas,-,INT,2,0,200,0,125000.0003,254,...,2,7,0,0,0,5,7,0,Analysis,1
3,0.000008,unas,-,INT,2,0,200,0,125000.0003,254,...,2,8,0,0,0,7,8,0,Analysis,1
4,0.000008,unas,-,INT,2,0,200,0,125000.0003,254,...,6,8,0,0,0,23,8,0,Analysis,1


In [3]:
# Count of all the data types 
data.dtypes.value_counts()

int64      29
float64    11
object      4
Name: count, dtype: int64

In [4]:
# Categorical variables
data.select_dtypes(exclude=np.number).columns

Index(['proto', 'service', 'state', 'attack_cat'], dtype='object')

In [5]:
# Numerical Variables
data.select_dtypes(include=np.number).columns

Index(['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl',
       'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit',
       'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean',
       'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src',
       'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
       'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd',
       'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'label'],
      dtype='object')

In [6]:
# Binarized categorical variables
print(data['is_sm_ips_ports'].value_counts())
print(data['is_ftp_login'].value_counts())
print(data['label'].value_counts())

is_sm_ips_ports
0    144478
1      1376
Name: count, dtype: int64
is_ftp_login
0    143488
1      2344
4        16
2         6
Name: count, dtype: int64
label
1    107077
0     38777
Name: count, dtype: int64


# Incorrect Entries
is_ftp_login has 22 incorrect entries as the values should be 1 If the ftp session is accessed by user and password else 0 (1 or 0)

In [7]:
# Filter out rows where 'is_ftp_login' is either 2 or 4
mode = data['is_ftp_login'].mode()[0]
data['is_ftp_login'] = data['is_ftp_login'].replace([2, 4], mode)
print(data['is_ftp_login'].value_counts())

is_ftp_login
0    143510
1      2344
Name: count, dtype: int64


# Missing Entries

In [8]:
data.isnull().sum().sum()

0

In [9]:
# Finding columns that have hidden missing entries listed as (-) instead of NaN/None
for col in data.columns:
    # Goes through all the columns and finds the real number of missing entries
    num_missing = len(data[data[col] == '-'])
    if num_missing > 0:
        ratio = num_missing / len(data[col])
        print(f"{col}: {num_missing} '-' entries ({round(ratio,2)}%)")

service: 69774 '-' entries (0.48%)


In [10]:
data['service'].value_counts()

service
-           69774
dns         46459
http        15847
smtp         4610
ftp-data     3727
ftp          2933
ssh          1253
pop3         1040
snmp           67
dhcp           61
ssl            52
irc            21
radius         10
Name: count, dtype: int64

In [11]:
# Replace (-) entries in service as null 
data['service'] = data['service'].replace('-', None)

# Duplicated Rows

In [12]:
# Find duplicated rows
duplicated_rows = data[data.duplicated()]
duplicated_rows.shape[0]

56646

In [13]:
# Calculate the ratio of the binary column in the duplicated rows
ratio = duplicated_rows['attack_cat'].value_counts(normalize=True)
# Print the ratio
print(ratio)

attack_cat
Generic           0.632331
Exploits          0.239187
Reconnaissance    0.052413
Fuzzers           0.035907
Normal            0.028475
Analysis          0.007167
Backdoor          0.003725
Shellcode         0.000741
Worms             0.000053
Name: proportion, dtype: float64


In [14]:
counts = duplicated_rows['label'].value_counts()
ratio = duplicated_rows['label'].value_counts(normalize=True)
print(counts, ratio)

label
1    55033
0     1613
Name: count, dtype: int64 label
1    0.971525
0    0.028475
Name: proportion, dtype: float64


There are a large number of duplicated rows (56633) but 97% (55021) come from the attack category. 

In [15]:
# Remove duplicate rows
data = data.drop_duplicates()
print(data.shape)

(89208, 44)


# One Hot Encoding

In [16]:
"""# OneHotEncoder
service_ = OneHotEncoder(sparse_output=False)
proto_ = OneHotEncoder(sparse_output=False)
state_ = OneHotEncoder(sparse_output=False)

# Fit and transform the columns directly
ohe_service = service_.fit_transform(data.service.values.reshape(-1, 1))
ohe_proto = proto_.fit_transform(data.proto.values.reshape(-1, 1))
ohe_state = state_.fit_transform(data.state.values.reshape(-1, 1))"""

'# OneHotEncoder\nservice_ = OneHotEncoder(sparse_output=False)\nproto_ = OneHotEncoder(sparse_output=False)\nstate_ = OneHotEncoder(sparse_output=False)\n\n# Fit and transform the columns directly\nohe_service = service_.fit_transform(data.service.values.reshape(-1, 1))\nohe_proto = proto_.fit_transform(data.proto.values.reshape(-1, 1))\nohe_state = state_.fit_transform(data.state.values.reshape(-1, 1))'

In [17]:
"""# Loop to concatenate the new columns into the original DataFrame
for col, ohe, categories in zip(['proto', 'service', 'state'], 
                                [ohe_proto, ohe_service, ohe_state],
                                [proto_.categories_, service_.categories_, state_.categories_]):
    # Convert the one-hot encoded data into a DataFrame
    tmp_df = pd.DataFrame(ohe, columns=[col + '_' + str(i) for i in categories[0]])
    tmp_df.index = data.index
    data = data.drop(col, axis=1)  # drop original column
    data = pd.concat([data, tmp_df], axis=1)  # add new columns
data.shape"""

"# Loop to concatenate the new columns into the original DataFrame\nfor col, ohe, categories in zip(['proto', 'service', 'state'], \n                                [ohe_proto, ohe_service, ohe_state],\n                                [proto_.categories_, service_.categories_, state_.categories_]):\n    # Convert the one-hot encoded data into a DataFrame\n    tmp_df = pd.DataFrame(ohe, columns=[col + '_' + str(i) for i in categories[0]])\n    tmp_df.index = data.index\n    data = data.drop(col, axis=1)  # drop original column\n    data = pd.concat([data, tmp_df], axis=1)  # add new columns\ndata.shape"

# Label Encoding
Turning the target variable attack_cat into a numerical format to be able to perform correlation calculations and model training

In [18]:
# Create an instance of LabelEncoder
label_encoder = LabelEncoder()

# Assuming 'train' is your DataFrame and 'attack_cat' is the categorical column
data['attack_cat_encoded'] = label_encoder.fit_transform(data['attack_cat'])

In [19]:
"""#Splitting the data to X,Y,Z
X_train, Y_train, Z_train = data.drop(columns=['attack_cat','label'], axis=1), data['attack_cat'], data['label']"""

"#Splitting the data to X,Y,Z\nX_train, Y_train, Z_train = data.drop(columns=['attack_cat','label'], axis=1), data['attack_cat'], data['label']"

In [20]:
data.to_csv("/Users/tristangarcia/desktop/Network Traffic Classification/data/train_clean.csv", index=False)

In [21]:
print(data.columns)

Index(['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes',
       'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss',
       'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin',
       'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth',
       'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm',
       'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label',
       'attack_cat_encoded'],
      dtype='object')
